# License Plate Detection — Model 2: Faster R-CNN
**CMPS 261 — Machine Learning Project**

Faster R-CNN is a two-stage detector:
1. **Region Proposal Network (RPN)** — proposes candidate regions that may contain an object
2. **Detection head** — classifies and refines each proposed region

This notebook runs **both locally and on Google Colab**. On Colab, select Runtime → Change runtime type → T4 GPU.

## 1. Environment Setup

In [ ]:
import sys, os, json, random, time

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on: {"Google Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', 'pycocotools', '-q'], check=True)

    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile
    zip_path = '/content/drive/MyDrive/license_plate_data.zip'
    if not os.path.exists('/content/data/yolo'):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/')
        print('Extracted dataset.')

    TRAIN_IMG = '/content/data/yolo/images/train'
    VAL_IMG   = '/content/data/yolo/images/val'
    TEST_IMG  = '/content/data/yolo/images/test'
    TRAIN_LBL = '/content/data/yolo/labels/train'
    VAL_LBL   = '/content/data/yolo/labels/val'
    TEST_LBL  = '/content/data/yolo/labels/test'
    MODEL_SAVE_PATH = '/content/fasterrcnn_best.pth'
    RESULTS_DIR     = '/content/results'
else:
    sys.path.append('..')
    TRAIN_IMG = '../data/yolo/images/train'
    VAL_IMG   = '../data/yolo/images/val'
    TEST_IMG  = '../data/yolo/images/test'
    TRAIN_LBL = '../data/yolo/labels/train'
    VAL_LBL   = '../data/yolo/labels/val'
    TEST_LBL  = '../data/yolo/labels/test'
    MODEL_SAVE_PATH = '../models/fasterrcnn_best.pth'
    RESULTS_DIR     = '../results'

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

import torch
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

DEVICE = (torch.device('cuda')  if torch.cuda.is_available()  else
          torch.device('mps')   if torch.backends.mps.is_available() else
          torch.device('cpu'))
print(f'Device: {DEVICE}' + (f'  ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))

## 2. Dataset & DataLoaders

Reads pre-split YOLO `.txt` labels and converts to `xyxy` format.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as F
from tqdm import tqdm

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.samples = []
        for lbl_file in sorted(os.listdir(lbl_dir)):
            if not lbl_file.endswith('.txt'): continue
            stem = os.path.splitext(lbl_file)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, os.path.join(lbl_dir, lbl_file)))
                    break

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        boxes = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                _, cx, cy, w, h = map(float, parts[:5])
                xmin = max(0.0, (cx-w/2)*W); ymin = max(0.0, (cy-h/2)*H)
                xmax = min(float(W),(cx+w/2)*W); ymax = min(float(H),(cy+h/2)*H)
                if xmax > xmin and ymax > ymin: boxes.append([xmin,ymin,xmax,ymax])
        if not boxes: boxes = [[0.0,0.0,1.0,1.0]]
        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        return F.to_tensor(img), {
            'boxes': boxes, 'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:,3]-boxes[:,1])*(boxes[:,2]-boxes[:,0]),
            'iscrowd': torch.zeros(len(boxes), dtype=torch.int64),
        }

def collate_fn(batch): return tuple(zip(*batch))

train_ds = LicensePlateDataset(TRAIN_IMG, TRAIN_LBL)
val_ds   = LicensePlateDataset(VAL_IMG,   VAL_LBL)
test_ds  = LicensePlateDataset(TEST_IMG,  TEST_LBL)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

workers = 2 if IN_COLAB else 0
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=workers)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=workers)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=workers)

## 3. Build the Model

Pretrained ResNet-50 FPN v2 backbone. We replace the box predictor head for our 1-class problem (background + licence plate).

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model   = fasterrcnn_resnet50_fpn_v2(weights=weights)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
model.to(DEVICE)

params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
print('Model ready.')

## 4. Training Loop

In [ ]:
NUM_EPOCHS = 30
best_val   = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    model.train()
    total_train = 0
    for images, targets in tqdm(train_loader, desc=f'Epoch {epoch} train', leave=False):
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss = sum(model(images, targets).values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_train += loss.item()
    train_loss = total_train / len(train_loader)

    model.train()
    total_val = 0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc=f'Epoch {epoch} val  ', leave=False):
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            total_val += sum(model(images, targets).values()).item()
    val_loss = total_val / len(val_loader)

    scheduler.step()
    train_losses.append(train_loss); val_losses.append(val_loss)

    flag = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH); flag = ' <- best'

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | {time.time()-t0:.0f}s{flag}')

print('Training complete!')

## 5. Loss Curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Faster R-CNN — Training & Validation Loss')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fasterrcnn_loss_curve.png'), dpi=150)
plt.show()

## 6. Evaluate on Test Set

In [ ]:
import numpy as np

def compute_iou(a, b):
    xA,yA = max(a[0],b[0]),max(a[1],b[1])
    xB,yB = min(a[2],b[2]),min(a[3],b[3])
    inter = max(0,xB-xA)*max(0,yB-yA)
    return inter/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter+1e-6)

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()

tp, fp, fn = 0, 0, 0
iou_scores = []
CONF_THRESH = 0.5

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc='Evaluating'):
        images = [img.to(DEVICE) for img in images]
        preds  = model(images)
        for pred, target in zip(preds, targets):
            gt_boxes   = target['boxes'].numpy()
            pred_boxes = pred['boxes'][pred['scores'] >= CONF_THRESH].cpu().numpy()
            matched = set()
            for pb in pred_boxes:
                best_iou, best_j = 0, -1
                for j, gb in enumerate(gt_boxes):
                    iou = compute_iou(pb, gb)
                    if iou > best_iou: best_iou, best_j = iou, j
                if best_iou >= 0.5 and best_j not in matched:
                    tp += 1; matched.add(best_j); iou_scores.append(best_iou)
                else: fp += 1
            fn += len(gt_boxes) - len(matched)

precision = tp/(tp+fp+1e-6); recall = tp/(tp+fn+1e-6)
f1       = 2*precision*recall/(precision+recall+1e-6)
mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0

print(f'\nTest Results (conf>={CONF_THRESH})')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1        : {f1:.4f}')
print(f'  Mean IoU  : {mean_iou:.4f}')

## 7. Visualise Predictions

In [ ]:
import matplotlib.patches as patches

sample_indices = random.sample(range(len(test_ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

with torch.no_grad():
    for ax, idx in zip(axes, sample_indices):
        img_tensor, target = test_ds[idx]
        pred = model([img_tensor.to(DEVICE)])[0]
        ax.imshow(img_tensor.permute(1,2,0).numpy())
        for box in target['boxes']:
            x1,y1,x2,y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='red',facecolor='none'))
        for box, score in zip(pred['boxes'], pred['scores']):
            if score < CONF_THRESH: continue
            x1,y1,x2,y2 = box.cpu().tolist()
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='lime',facecolor='none'))
            ax.text(x1,y1-4,f'{score:.2f}',color='lime',fontsize=8,bbox=dict(facecolor='black',alpha=0.4,pad=1))
        ax.axis('off')

plt.suptitle('Faster R-CNN — Test Predictions (red=GT, lime=pred)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fasterrcnn_predictions.png'), dpi=150)
plt.show()

## 8. Save Metrics

In [ ]:
metrics = {
    'model'    : 'Faster R-CNN (ResNet50-FPN v2)',
    'precision': round(precision, 4),
    'recall'   : round(recall,    4),
    'f1'       : round(f1,        4),
    'mean_iou' : round(mean_iou,  4),
}
metrics_path = os.path.join(RESULTS_DIR, 'fasterrcnn_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

## 9. Download Weights (Colab only)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(MODEL_SAVE_PATH)
    files.download(metrics_path)
    print('Downloading...')
else:
    print(f'Weights saved at: {MODEL_SAVE_PATH}')